In [ ]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path


INPUT_PATH = "culture_markets_with_news_full_clean.csv"
OUTPUT_CSV = "culture_outcome_rl_format.csv"
OUTPUT_PARQUET = "culture_outcome_rl_format.parquet"


BASE_PROMPT = """You are an expert superforecaster, familiar with Structured Analytic Techniques as well as Superforecasting by Philip Tetlock and related work.

Predict the probability that the following question will be resolved as true/yes. You MUST give a probability estimate between 0 and 1 UNDER ALL CIRCUMSTANCES.

Question: {question}

Question close date: {close_date}

Use the following news articles as context. Consider source quality, timing, relevance, and whether the articles provide direct evidence about the resolution criteria.

{news_block}

Return your final answer as a probability between 0 and 1."""

# чистим текст от мусора
def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""

    text = str(text)

    # убираем markdown-ссылки
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)

    # убираем markdown-картинки
    text = re.sub(r"!\[[^\]]*\]\([^)]+\)", " ", text)

    # убираем лишние спецсимволы и пробелы
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text

# ограничиваем длину одной новости
def truncate_text(text: str, max_chars: int = 2500) -> str:
    text = clean_text(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rsplit(" ", 1)[0] + "..."


def parse_news_context(news_context):
    if pd.isna(news_context):
        return []

    try:
        news = json.loads(news_context)
        if isinstance(news, list):
            return news
        return []
    except Exception:
        return []


def normalize_date(x):
    if pd.isna(x):
        return None

    dt = pd.to_datetime(x, errors="coerce", utc=True)
    if pd.isna(dt):
        return None

    return dt.date().isoformat()

# отбираем новости по prediction_date
def filter_news_by_date(news, cutoff_date):
    if cutoff_date is None:
        return news

    cutoff_dt = pd.to_datetime(cutoff_date, errors="coerce", utc=True)
    if pd.isna(cutoff_dt):
        return news

    filtered = []

    for item in news:
        pub_date = item.get("publishedDate")
        pub_dt = pd.to_datetime(pub_date, errors="coerce", utc=True)

        if pd.isna(pub_dt):
            filtered.append(item)
        elif pub_dt <= cutoff_dt:
            filtered.append(item)

    return filtered


def format_news_block(news, max_articles=8, max_chars_per_article=2500):
    if not news:
        return "No relevant news articles were found."

    parts = []

    for i, item in enumerate(news[:max_articles], start=1):
        title = clean_text(item.get("title", "Untitled"))
        url = item.get("url", "")
        published = item.get("publishedDate", "")
        author = clean_text(item.get("author", ""))
        text = truncate_text(item.get("text", ""), max_chars=max_chars_per_article)

        article = f"""Article [{i}]
Title: {title}
Published date: {published}
Author: {author}
URL: {url}
Text: {text}"""

        parts.append(article)

    return "\n\n".join(parts)


def build_prompt(row, cutoff_mode="prediction_date"):
    question = row["question"]
    close_date = normalize_date(row["endDate"])

    news = parse_news_context(row["news_context"])

    if cutoff_mode == "prediction_date":
        news = filter_news_by_date(news, row.get("prediction_date"))
    elif cutoff_mode == "endDate":
        news = filter_news_by_date(news, row.get("endDate"))

    news_block = format_news_block(
        news,
        max_articles=8,
        max_chars_per_article=2500
    )

    return BASE_PROMPT.format(
        question=question,
        close_date=close_date,
        news_block=news_block
    )


def get_optional_column(df, candidates, default=np.nan):
    for col in candidates:
        if col in df.columns:
            return df[col]
    return default


def convert_to_outcome_rl_format(input_path: str):
    df = pd.read_csv(input_path)

    out = pd.DataFrame()

    # ID вопроса
    out["question_id"] = "culture_" + df["row_index"].astype(str)

    # текст вопроса
    out["question_text"] = df["question"].astype(str)

    # дата закрытия вопроса
    out["question_close_date"] = df["endDate"].apply(normalize_date)

    # volume
    out["volume"] = get_optional_column(
        df,
        candidates=["volume", "liquidity", "market_volume", "trade_volume"],
        default=np.nan
    )

    # resolution
    out["resolution"] = df["final_outcome"].astype(int)

    # prompt с новостями
    out["prompt"] = df.apply(
        lambda row: build_prompt(row, cutoff_mode="prediction_date"),
        axis=1
    )

    # polymarket prediction
    out["prediction_polymarket"] = get_optional_column(
        df,
        candidates=[
            "prediction_polymarket",
            "polymarket_prediction",
            "market_probability",
            "probability",
            "last_price",
            "lastTradePrice"
        ],
        default=np.nan
    )

    # O1 prediction (появится только после прогона модели)
    out["prediction_o1"] = get_optional_column(
        df,
        candidates=["prediction_o1", "o1_prediction"],
        default=np.nan
    )

    # ReMax prediction
    out["prediction_remax"] = get_optional_column(
        df,
        candidates=["prediction_remax", "remax_prediction"],
        default=np.nan
    )

    # Reasoning ReMax
    out["reasoning_remax"] = get_optional_column(
        df,
        candidates=["reasoning_remax", "remax_reasoning"],
        default=np.nan
    )

    # Контроль порядка колонок
    target_columns = [
        "question_id",
        "question_text",
        "question_close_date",
        "volume",
        "resolution",
        "prompt",
        "prediction_polymarket",
        "prediction_o1",
        "prediction_remax",
        "reasoning_remax",
    ]

    out = out[target_columns]

    return out


result = convert_to_outcome_rl_format(INPUT_PATH)

result.to_csv(OUTPUT_CSV, index=False)
try:
    result.to_parquet(OUTPUT_PARQUET, index=False)
except Exception as e:
    print("Parquet не сохранился. Установи pyarrow:")
    print("pip install pyarrow")
    print("Ошибка:", e)

print(result.shape)
print(result.head(2))